# SlideScribe — illustrated transcripts from conference recordings

Upload a screen-shared meeting recording, get back a PDF where each slide image sits
directly above the words spoken while it was on screen.

**How to run:** `Runtime` → `Run all`, then follow the prompts in Cell 2.

> **Turn on the GPU first:** `Runtime` → `Change runtime type` → `T4 GPU`.
> It still works on CPU, just slower. Cell 1 tells you which one you got.

---
## Cell 1 — Setup

Installs the package and checks your runtime. Takes about two minutes.

> If this cell reports that the package is not importable, run it once more. Colab occasionally restarts the kernel after installing, and the second run picks up where it left off.

In [ ]:
#@title Install SlideScribe { display-mode: "form" }

# Colab sometimes restarts the kernel silently after a large install, which
# wipes the session and makes the import in a later cell fail. Verifying the
# import here means this cell fails loudly rather than the next one failing
# mysteriously.

!pip install -q git+https://github.com/nishitghosh81/slidescribe.git litellm

import importlib, sys, textwrap

try:
    importlib.invalidate_caches()
    import slidescribe
    print(f"SlideScribe {slidescribe.__version__} installed.")
except ModuleNotFoundError:
    print(textwrap.dedent("""
        The install finished but the package is not importable, which means
        Colab restarted the kernel partway through.

        Fix: run this cell once more. Nothing needs reinstalling, so it will
        be quick and the import will succeed.
    """))
    raise SystemExit("Re-run this cell.")

import torch

if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
    print("A 45-minute video will take roughly 4-8 minutes.")
    SUGGESTED_WHISPER = "small"
else:
    print("No GPU on this runtime — running on CPU.")
    print(textwrap.dedent("""
        This works, but a 45-minute video may take 30-60 minutes.
        For a faster run: Runtime > Change runtime type > T4 GPU, then Run all again.
    """))
    SUGGESTED_WHISPER = "base"

print(f"\nSuggested Whisper model for this runtime: {SUGGESTED_WHISPER}")

---
## Cell 2 — Configure and run

Fill in the form below, then run the cell. Everything is optional except the video.

### About the model key

SlideScribe works with **no key at all** — transcription, slide detection and the PDF
are all local. A key adds three things: smarter slide detection (it can tell a real
slide change from a moving cursor or an animation), a title on each slide, and a
summary with action items.

Your key is used only for this session and never stored.

**Model strings** — put the provider prefix in, LiteLLM routes the rest:

| Provider | `MODEL` | Key from |
|---|---|---|
| OpenAI | `gpt-4o` | platform.openai.com |
| Anthropic | `claude-sonnet-4-5` | console.anthropic.com |
| Google | `gemini/gemini-2.0-flash` | aistudio.google.com |
| Groq (fast, free tier) | `groq/llama-3.3-70b-versatile` | console.groq.com |
| OpenRouter (any model) | `openrouter/anthropic/claude-sonnet-4-5` | openrouter.ai |
| Ollama / vLLM / local | `ollama/llava` | leave key blank, set `API_BASE` |

For slide titles and slide arbitration the model needs to **accept images**. Text-only
models still give you the summary — SlideScribe detects this automatically and falls
back rather than failing.

In [ ]:
#@title Settings { display-mode: "form" }

#@markdown ### Model (optional — leave blank to run with no LLM)
MODEL = ""  #@param {type:"string"}
API_KEY = ""  #@param {type:"string"}
API_BASE = ""  #@param {type:"string"}

#@markdown ### Transcription
WHISPER_MODEL = "small"  #@param ["tiny", "base", "small", "medium", "large-v3"]
LANGUAGE = ""  #@param {type:"string"}

#@markdown ### Slide detection
#@markdown Lower `HASH_THRESHOLD` and higher `SSIM_THRESHOLD` catch more changes.
HASH_THRESHOLD = 6  #@param {type:"slider", min:3, max:16, step:1}
SSIM_THRESHOLD = 0.95  #@param {type:"slider", min:0.85, max:0.99, step:0.01}
MIN_SLIDE_SECONDS = 4.0  #@param {type:"number"}
#@markdown Crop to the shared screen if a webcam tile sits beside it, as `left,top,right,bottom`
#@markdown fractions. Example: `0,0,0.75,1` keeps the left three quarters. Blank uses the whole frame.
CROP = ""  #@param {type:"string"}

#@markdown ### Output
SUMMARY = True  #@param {type:"boolean"}
SLIDE_TITLES = True  #@param {type:"boolean"}
CLEAN_TRANSCRIPT = False  #@param {type:"boolean"}

print("Settings loaded. Run the next cell to upload and process.")

### Now upload and process

Pick where your video comes from. **Google Drive is much more reliable for anything
over ~200 MB** — browser uploads of large files often stall.

In [ ]:
#@title Upload and run { display-mode: "form" }

SOURCE = "Upload from computer"  #@param ["Upload from computer", "Google Drive path", "Sample video"]
DRIVE_PATH = ""  #@param {type:"string"}

import os, textwrap
from slidescribe import Config, LLMConfig, process_video
from slidescribe.llm import LLMClient

# --- get the video -------------------------------------------------
if SOURCE == "Upload from computer":
    from google.colab import files
    print("Choose your video file...")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file chosen. Run this cell again.")
    video_path = list(uploaded.keys())[0]

elif SOURCE == "Google Drive path":
    from google.colab import drive
    drive.mount("/content/drive")
    video_path = DRIVE_PATH.strip()
    if not video_path:
        raise SystemExit("Set DRIVE_PATH above, e.g. /content/drive/MyDrive/meeting.mp4")
    if not os.path.exists(video_path):
        raise SystemExit(f"Not found: {video_path}")

else:
    print("Building a 48-second sample recording...")
    import cv2, numpy as np
    W, H, FPS = 1280, 720, 10
    vw = cv2.VideoWriter("sample.mp4", cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
    for i, (txt, bg) in enumerate([
        ("Q3 Revenue Overview", (30, 40, 60)), ("Market Expansion", (60, 30, 40)),
        ("Cost Structure", (35, 55, 35)), ("Next Steps", (50, 45, 25))]):
        base = np.full((H, W, 3), bg, np.uint8)
        cv2.putText(base, txt, (90, 220), cv2.FONT_HERSHEY_SIMPLEX, 2.2, (240,)*3, 4)
        for b in range(3):
            cv2.putText(base, f"- point {b+1} of section {i+1}", (110, 340+b*70),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.1, (200, 205, 210), 2)
        for f in range(12 * FPS):
            fr = base.copy()
            cv2.circle(fr, (300 + int(200*np.sin(f/8)), 600), 9, (255,)*3, -1)
            vw.write(fr)
    vw.release()
    video_path = "sample.mp4"
    print("Note: the sample has no audio, so the transcript will be empty.")

size_mb = os.path.getsize(video_path) / 1e6
print(f"\nProcessing: {video_path} ({size_mb:.0f} MB)")

# --- build config --------------------------------------------------
crop = None
if CROP.strip():
    parts = [float(x) for x in CROP.split(",")]
    if len(parts) == 4:
        crop = tuple(parts)
        print(f"Cropping to region: {crop}")
    else:
        print("CROP needs four numbers — ignoring it and using the full frame.")

llm_cfg = LLMConfig(
    model=MODEL.strip() or None,
    api_key=API_KEY.strip() or None,
    api_base=API_BASE.strip() or None,
)

# Validate the key before committing to a long run.
if llm_cfg.model:
    print(f"\nChecking {llm_cfg.model}...")
    status = LLMClient(llm_cfg).check()
    if status["ok"]:
        print(f"  Connected. Vision support: {'yes' if status['vision'] else 'no (text only)'}")
        if not status["vision"]:
            print("  Slide titles will be skipped; the summary still works.")
    else:
        print(f"  Could not connect: {status['detail']}")
        print("  Continuing without the model — the PDF will still be built.")
        llm_cfg = LLMConfig()
else:
    print("\nNo model configured — running the local pipeline only.")

config = Config(
    whisper_model=WHISPER_MODEL,
    language=LANGUAGE.strip() or None,
    hash_threshold=HASH_THRESHOLD,
    ssim_threshold=SSIM_THRESHOLD,
    min_slide_seconds=MIN_SLIDE_SECONDS,
    crop=crop,
    llm=llm_cfg,
    caption_slides=SLIDE_TITLES,
    summarize=SUMMARY,
    clean_transcript=CLEAN_TRANSCRIPT,
)

result = process_video(video_path, config=config)

print(textwrap.dedent(f"""
    ------------------------------------------
    {len(result.slides)} slides · {len(result.segments)} transcript segments
    Finished in {result.seconds:.0f} seconds
    ------------------------------------------"""))

from google.colab import files as _f
_f.download(result.pdf_path)
print(f"Downloading {result.pdf_path}")

---
## If the results need adjusting

**Too many near-identical slides** — the detector is firing on animations or video.
Raise `HASH_THRESHOLD` to 8-10, lower `SSIM_THRESHOLD` to 0.92, or raise
`MIN_SLIDE_SECONDS` to 8. Adding a vision model fixes this properly, since it can tell
an animated build from a genuine new slide.

**Slides were missed** — lower `HASH_THRESHOLD` to 4 and raise `SSIM_THRESHOLD` to 0.97.

**A webcam tile is triggering captures** — set `CROP` to the shared-screen region.
For a screen share on the left and faces on the right, `0,0,0.75,1` works well.

**Transcript quality is poor** — move up to `medium` or `large-v3` (GPU strongly
recommended), and set `LANGUAGE` if the recording is not in English.

**The session disconnected** — Colab drops idle sessions after about 90 minutes.
Keep the tab open, or use a shorter Whisper model.

---

Repo: `github.com/nishitghosh81/slidescribe` · MIT licensed. Issues and PRs welcome.